In [ ]:
#import + session DB + subgraph storage obj

import pandas as pd

from HOGDB.graph.subgraph import Subgraph          # constructeur Subgraph (API HO-GDB)
from HOGDB.graph.graph_with_subgraph_storage import GraphwithSubgraphStorage
from HOGDB.graph.graph_with_tuple_storage import *
from HOGDB.graph.edge import Edge
from HOGDB.db.neo4j import Neo4jDatabase

# INIT DB

db = Neo4jDatabase()
gs = GraphwithSubgraphStorage(db)

In [ ]:

#add taggedComment Subgraphs

SUBGRAPH_BATCH = 1000

print("===Insert taggedComment subgraphs ")

buffer = []
count = 0

for (s_id, t_id), e in edge_cache_copy.items():
    sg = Subgraph(
        subgraph_nodes=[
            comment_cache[s_id],
            tag_cache[t_id]
        ],
        subgraph_edges=[e],
        labels=[Label("taggedComment")],
        properties=[
            Property("id", str, f"tc_{s_id}_{t_id}")
        ]
    )

    buffer.append(sg)

    if len(buffer) == SUBGRAPH_BATCH:
        for s in buffer:
            gs.add_subgraph(s)
        count += len(buffer)
        buffer.clear()
        print(f"  inserted {count} subgraphs")

# flush last batch
if buffer:
    for s in buffer:
        gs.add_subgraph(s)
    count += len(buffer)

print(f"== DONE: {count} subgraphs inserted ==")





In [ ]:
# add forumMembership Subgraphs

SUBGRAPH_BATCH = 1000

print("=== Insert forumMembership subgraphs ===")

buffer = []
count = 0

# group persons and edges by forum
forum_members = {}

for (f_id, p_id), e in edgefm_cache.items():

    if f_id not in forum_members:
        forum_members[f_id] = {
            "persons": [],
            "edges": []
        }

    forum_members[f_id]["persons"].append(person_cache[p_id])
    forum_members[f_id]["edges"].append(e)


# create subgraphs
for f_id, data in forum_members.items():

    forum_node = forum_cache[f_id]
    persons = data["persons"]
    edges = data["edges"]

    sg = Subgraph(
        subgraph_nodes=[forum_node] + persons,
        subgraph_edges=edges,
        labels=[Label("forumMembership")],
        properties=[
            Property("id", str, f"fm_{f_id}"),
            Property("nbMember", int, len(persons))
        ]
    )

    buffer.append(sg)

    if len(buffer) == SUBGRAPH_BATCH:
        for s in buffer:
            gs.add_subgraph(s)

        count += len(buffer)
        buffer.clear()
        print(f"  inserted {count} subgraphs")


# flush last batch
if buffer:
    for s in buffer:
        gs.add_subgraph(s)
    count += len(buffer)

print(f"== DONE: {count} forumMembership subgraphs inserted ==")

In [ ]:
# CLOSE
gs.close_connection()